# 2019 Transactional Sales Analysis | Professional Refactor

**Workflow:** ingest monthly CSV/SQL transactions → standardize → clean → engineer sales features → explore revenue and products → test weekday differences → export.

**Setup:** place `Sales_<Month>_2019.csv` files in `data/` (or set `SALES_DATA_DIR`). If July–December CSVs are unavailable, configure SQL Server credentials in `.env`. Charts and computed results require access to the underlying data.

## 1. Business Understanding

### 1.1. Introduction

In this project, we hope to create and implement a from start to finish business intelligence system that uses transactional data from 2019 to provide relevant insights to our client. The client has gathered data throughout the year but has not used it effectively to boost sales and enhance operational efficiency. Our goal is to thoroughly analyse the data and provide actionable insights that will help the customer identify chances for growth and optimisation.

## Project Description

This project involves the analysis of sales data collected by a client for the year 2019. The goal is to uncover insights that can help the client optimize their sales strategy, increase revenue, and identify operational inefficiencies. The analysis will follow the CRISP-DM framework, answering key business questions such as:

### Hypothesis

- **H0:** The distribution of quantity ordered is the same across weekdays.
- **H1:** At least one weekday has a different distribution of quantity ordered.

### Analytical Questions

1. How much money did we make this year? 

2. Can we identify any seasonality in the sales? 

3. What are our best and worst-selling products? 

4. How do sales compare to previous months or weeks? 

5. Which cities are our products delivered to most? 

6. How do product categories compare in revenue generated and quantities ordered? 

7. You are required to show additional details from your findings in your data. 


### Getting Started

### Importation of libraries

In [ ]:
# Core dependencies for transactional sales analysis.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal

sns.set_theme(style="whitegrid")
SALES_YEAR = 2019
MONTH_NAMES = ("January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December")
SALES_COLUMNS = ("Order_ID", "Product", "Quantity_Ordered", "Price_Each", "Order_Date", "Purchase_Address")
DATA_DIR = Path(os.getenv("SALES_DATA_DIR", "data"))
OUTPUT_DIR = Path(os.getenv("SALES_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def standardize_sales_columns(sales_frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize CSV/SQL column names to a shared transactional schema."""
    normalized = sales_frame.copy()
    normalized.columns = normalized.columns.str.strip().str.replace(" ", "_", regex=False)
    missing = set(SALES_COLUMNS) - set(normalized.columns)
    if missing:
        raise ValueError(f"Missing sales columns: {sorted(missing)}")
    return normalized.loc[:, list(SALES_COLUMNS)]


# Data Loading


In [ ]:
# Read the January–June extracts; override SALES_DATA_DIR for a custom location.
first_half_files = [DATA_DIR / f"Sales_{month}_{SALES_YEAR}.csv" for month in MONTH_NAMES[:6]]
missing_csv_files = [str(path) for path in first_half_files if not path.is_file()]
if missing_csv_files:
    raise FileNotFoundError(f"Missing first-half CSV files: {missing_csv_files}")
first_half_sales = pd.concat(
    (standardize_sales_columns(pd.read_csv(path)) for path in first_half_files),
    ignore_index=True,
)
first_half_sales.head()


#### Loading the second data from SQL database

In [ ]:
# SQL credentials are optional when all twelve monthly CSV files are available.
# Store SERVER, DATABASE, USERNAME and PASSWORD in a local .env file.
try:
    from dotenv import dotenv_values
    sql_credentials = dotenv_values(".env")
except ImportError:
    sql_credentials = {}
sql_credentials = {key: os.getenv(key) or sql_credentials.get(key) for key in ("SERVER", "DATABASE", "USERNAME", "PASSWORD")}


In [ ]:
# Prefer July–December CSV exports; otherwise load the corresponding SQL Server tables.
second_half_files = [DATA_DIR / f"Sales_{month}_{SALES_YEAR}.csv" for month in MONTH_NAMES[6:]]
if all(path.is_file() for path in second_half_files):
    second_half_sales = pd.concat(
        (standardize_sales_columns(pd.read_csv(path)) for path in second_half_files),
        ignore_index=True,
    )
else:
    missing_keys = [key for key, value in sql_credentials.items() if not value]
    if missing_keys:
        raise FileNotFoundError(
            "July–December CSV files are incomplete and SQL credentials are missing: "
            + ", ".join(missing_keys)
        )
    try:
        import pyodbc
    except ImportError as exc:
        raise ImportError("Install pyodbc to load second-half sales from SQL Server") from exc
    sql_connection_string = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={sql_credentials['SERVER']};DATABASE={sql_credentials['DATABASE']};"
        f"UID={sql_credentials['USERNAME']};PWD={sql_credentials['PASSWORD']};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    with pyodbc.connect(sql_connection_string) as sql_connection:
        second_half_sales = pd.concat(
            [standardize_sales_columns(pd.read_sql_query(
                f"SELECT Order_ID, Product, Quantity_Ordered, Price_Each, Order_Date, Purchase_Address "
                f"FROM dbo.Sales_{month}_{SALES_YEAR}", sql_connection
            )) for month in MONTH_NAMES[6:]],
            ignore_index=True,
        )
second_half_sales.head()


### 3. Data Preparation
3.1. Data Cleaning

3.1.1. Standardize columns for both datasets.

In [ ]:
first_half_sales.columns

In [ ]:
second_half_sales.columns

- Column names in the first half include 'Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Order Date', and 'Purchase Address', whereas the second half has 'Order_ID', 'Product', 'Quantity_Ordered', 'Price_Each', 'Order_Date', and 'Purchase_Address'. First, we need to standardise the column names.

In [ ]:
first_half_sales = standardize_sales_columns(first_half_sales)
first_half_sales.columns

### 3.1.2. Concatenate Dataset

In [ ]:
annual_sales_raw = pd.concat([first_half_sales, second_half_sales], ignore_index=True)
annual_sales_raw.head()

### 3.1.3. Check data structure

In [ ]:
annual_sales_raw.info()

- The table above shows the amount of values in each column, as well as the data type.
- The datatypes for Order ID, Quantity Ordered, Price Each, and Order Date are incorrect and must be corrected.
- To establish an accurate dataset for this study, additional characteristics will be generated while others will be removed.

In [ ]:
f"Loaded {annual_sales_raw.shape[0]:,} transactions across {annual_sales_raw.shape[1]} columns."

### 3.1.4. Handle missing values

In [ ]:
annual_sales_raw.isna().sum()

In [ ]:
incomplete_sales_rows = annual_sales_raw.loc[annual_sales_raw.isna().any(axis=1)]
incomplete_sales_rows.head(50)

In [ ]:
incomplete_sales_rows.tail(20)

- Because the rows with missing values in the dataset have all columns that are missing, we cannot impute the missing values, hence we remove the null values from the dataset.

In [ ]:
annual_sales_raw = annual_sales_raw.dropna(subset=SALES_COLUMNS).copy()
annual_sales_raw.isna().sum()

### 3.1.5 Remove duplicates

In [ ]:
f"Duplicate transactions: {annual_sales_raw.duplicated().sum():,}"

In [ ]:
duplicate_sales_rows = annual_sales_raw.loc[annual_sales_raw.duplicated()]
duplicate_sales_rows

The duplicated rows in the dataset will be dealt with by keeping the first occurence and dropping every other instance.

In [ ]:
annual_sales_raw = annual_sales_raw.drop_duplicates().copy()
f"Remaining duplicates: {annual_sales_raw.duplicated().sum():,}"

### 3.1.6. Check columns unique values

In [ ]:
annual_sales_raw["Product"].unique()

- The Product value in the Product column must be inspected to validate the value

In [ ]:
annual_sales_raw.loc[annual_sales_raw["Product"].eq("Product")]

This row in the dataset must be removed since it is invalid and contains incorrect values.

In [ ]:
annual_sales_raw = annual_sales_raw.loc[~annual_sales_raw["Product"].eq("Product")].copy()

In [ ]:
annual_sales_raw.loc[annual_sales_raw["Product"].eq("Product")]

In [ ]:
annual_sales_raw["Purchase_Address"].head()

- The purchasing address column contains numerous pieces of information that will be splitted for easier dataset analysis.

- Columns such as Street Name, City, State, and ZIP Code will be taken from this column, which will then be removed.

In [ ]:
annual_sales_raw["Order_Date"].unique()

From the output for the unique values in the order date column, we can see that there are dates other than 2019 that we will need to remove in order to focus our study on the required date alone.

### 3.1.7. Convert data types

In [ ]:
# Coerce malformed values rather than crashing on embedded headers or invalid dates.
annual_sales_raw["Order_Date"] = pd.to_datetime(annual_sales_raw["Order_Date"], errors="coerce")
for sales_numeric_column in ("Quantity_Ordered", "Price_Each"):
    annual_sales_raw[sales_numeric_column] = pd.to_numeric(annual_sales_raw[sales_numeric_column], errors="coerce")
annual_sales_raw = annual_sales_raw.dropna(subset=["Order_Date", "Quantity_Ordered", "Price_Each"]).copy()


In [ ]:
annual_sales_raw["Order_Date"].unique()

In [ ]:
annual_sales_raw.info()

### 3.2. Data Transformation
#### 3.2.1. Generate new features

In [ ]:
sales_2019 = annual_sales_raw.loc[annual_sales_raw["Order_Date"].dt.year.eq(SALES_YEAR)].copy()
sales_2019.head()

In [ ]:
sales_2019["Sales"] = sales_2019["Quantity_Ordered"] * sales_2019["Price_Each"]

In [ ]:
# Parse US purchase addresses as street, city and state/ZIP components.
address_parts = sales_2019["Purchase_Address"].str.split(",", n=2, expand=True)
sales_2019[["Street_Address", "City", "State_ZIP_Code"]] = address_parts.apply(lambda part: part.str.strip())
sales_2019.head()

In [ ]:
state_zip_parts = sales_2019["State_ZIP_Code"].str.extract(r"^\s*(?P<State>[A-Za-z]{2})\s+(?P<ZIP_Code>\d{5}(?:-\d{4})?)\s*$")
sales_2019[["State", "ZIP_Code"]] = state_zip_parts

In [ ]:
sales_2019 = sales_2019.drop(columns=["Purchase_Address", "State_ZIP_Code"])
sales_2019.head()

In [ ]:
sales_2019["Day_of_Week"] = sales_2019["Order_Date"].dt.day_name()

In [ ]:
# Vectorized product price bands avoid per-row Python loops.
sales_2019["Price_Tier"] = np.where(sales_2019["Price_Each"] > 99.99, "High Level", "Basic Level")

In [ ]:
sales_2019.head()

## 4. Exploratory Data Analysis (EDA)

#### 4.1. Summary statistics

#### 4.1.1. Analysis for numerical columns

In [ ]:
sales_2019.select_dtypes(include="number").describe().T

Iinsights from summary statistics
1. Order ID - The collection contains 185,686 order IDs, representing the total number of transactions.
- The order IDs range between 141,234 and 319,670, as indicated by the Min and Max figures.
- There are no missing values for order IDs because the count equals the total number of rows in the data set.

2. Ordered Quantity - The typical transaction includes 1.12 items.
- The standard deviation of approximately 0.44 suggests that the quantity ordered ranges somewhat around the mean.
- The least quantity ordered is one, and the greatest is nine, with the median and quartiles indicating that the majority of transactions are ordered in quantities of one.

3. Price Each
- The mean price of each item ordered is approximately 184.52, with a standard diviation of 332.84, indicating significant
variability in item prices.
- Item prices range from 2.99 to 1700.00.
- Most items have prices between 11.95 and 150.00, as indicated by the quartiles.

### 4.1.2. Analysis for categorical columns

In [ ]:
sales_2019.describe(include="object").T

- Using the descriptive statistics for the categorical columns provided above, we can calculate the number of values in the columns, the number of unique values, the most common values, and the frequency of the most common value.

## 4.2. Univariate Analysis

In [ ]:
daily_units_sold = sales_2019.groupby(sales_2019["Order_Date"].dt.date)["Quantity_Ordered"].sum()
ax = daily_units_sold.plot(figsize=(15, 6), title="Daily Units Sold — 2019")
ax.set(xlabel="Order date", ylabel="Units sold")
plt.tight_layout()
plt.show()

A few insights from the graph are:

- Steady Sales (Early 2019 to Mid-2019): From January to around July 2019, there is a relatively consistent pattern of sales with most days showing sales between 4 and 7 units.
 
- Sudden Drops (After Mid-2019): There are noticeable drops in sales starting in July, with periods of very low or no sales, followed by spikes.

- Regular Spikes (Late 2019): From September onwards, the sales trend shows sharp spikes, likely indicating seasonal or periodic sales events (e.g., promotions, holidays). These spikes occur in September, November, and December.

- End-of-Year Surge: The highest sales spike occurs around November/December, which could indicate holiday or end-of-year sales boosts.

The sales trend suggests a relatively stable first half of the year, followed by more volatile sales in the second half, likely driven by special events or holidays.

In [ ]:
product_revenue_rank = sales_2019.groupby("Product")["Sales"].sum().sort_values()
ax = product_revenue_rank.plot.barh(figsize=(12, 9), title="2019 Revenue by Product")
ax.set(xlabel="Revenue ($)", ylabel="Product")
plt.tight_layout()
plt.show()

 Here are some key insights:

- Top Performers:
Macbook Pro Laptop generates the highest revenue, significantly outperforming other products.
iPhone and ThinkPad Laptop follow closely behind as other major revenue contributors.

- Mid-Range Performers:
Products like Google Phone, 34in Ultrawide Monitor, 27in 4K Gaming Monitor, and Apple Airpods Headphones are mid-tier in terms of revenue, contributing a decent share.

- Low Performers:
AA Batteries (4-pack), AAA Batteries (4-pack), and Wired Headphones contribute the least revenue, indicating these might be low-margin or low-sales items.

This chart suggests that high-end electronics (laptops, phones, monitors) drive most of the revenue, while accessories and smaller items bring in less revenue.

## 4.3. Bivariate Analysis

In [ ]:
sales_numeric_correlations = sales_2019[["Quantity_Ordered", "Price_Each", "Sales"]].corr()

In [ ]:
sns.heatmap(sales_numeric_correlations, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Sales Metrics Correlation")
plt.tight_layout()
plt.show()

The correlation matrix shows the relationship between two variables: Quantity Ordered and Price Each.

- Quantity Ordered and Price Each have a weak negative correlation of -0.15. This suggests that as the price of an item increases, the quantity ordered tends to decrease slightly, though the relationship is not strong.
- The diagonal elements (1.00) indicate that each variable is perfectly correlated with itself, which is expected.

Overall, there is a small inverse relationship between the price of a product and the quantity ordered.

## 4.3. Multivariate Analysis

In [ ]:
sns.pairplot(sales_2019[["Quantity_Ordered", "Price_Each", "Sales"]].sample(n=min(2000, len(sales_2019)), random_state=42))
plt.show()

Here's a quick breakdown of insights:

- Quantity_Ordered vs. Sales: There is a strong positive correlation, meaning that as more products are ordered, sales increase.
- Price_Each vs. Sales: Another strong positive correlation, indicating that higher-priced products result in higher sales.
- Quantity_Ordered: Most orders are for smaller quantities (1-2 items), with fewer orders for larger quantities.
= Order_ID doesn't seem to show any meaningful relationship with other variables, as it's just a unique identifier.

These plots help identify trends and relationships in the data, such as how quantity and price affect total sales.

## 5. Answering Analytical Questions

#### 5.1. How much money did we make in 2019?

In [ ]:
f"Total 2019 sales revenue: ${sales_2019['Sales'].sum():,.2f}"

####  5.2. Can we identify any seasonality in the sales?

To identify seasonality in sales, we'll analyze the sales data over time, specifically by month. We'll plot the total sales for each month over the entire duration of the dataset and look for recurring patterns or trends.

1. Group the sales data by month.
2. Calculate the total sales for each month.
3. Plot the total sales for each month.

In [ ]:
sales_2019["Order_Month"] = sales_2019["Order_Date"].dt.month
monthly_revenue = sales_2019.groupby("Order_Month")["Sales"].sum().reindex(range(1, 13), fill_value=0)

In [ ]:
ax = monthly_revenue.plot(marker="o", figsize=(10, 6), title="Monthly Revenue — 2019")
ax.set(xlabel="Month", ylabel="Revenue ($)", xticks=range(1, 13))
plt.tight_layout()
plt.show()

Monthly revenue can reveal seasonality, but one year alone is insufficient to establish recurring seasonal patterns.

#### 5.3 What are our best and worst-selling products?

To determine the best and worst-selling products, we will tally the number of units sold and the total money earned for each product. Then we will identify the goods with the highest and lowest overall quantity and income.

In [ ]:
product_performance = sales_2019.groupby("Product").agg(units_sold=("Quantity_Ordered", "sum"), revenue=("Sales", "sum"))
best_products_by_units = product_performance.nlargest(5, "units_sold")
best_products_by_revenue = product_performance.nlargest(5, "revenue")
worst_products_by_units = product_performance.nsmallest(5, "units_sold")
worst_products_by_revenue = product_performance.nsmallest(5, "revenue")

In [ ]:
print("Top 5 products by units:\n", best_products_by_units)
print("Top 5 products by revenue:\n", best_products_by_revenue)
print("Bottom 5 products by units:\n", worst_products_by_units)
print("Bottom 5 products by revenue:\n", worst_products_by_revenue)

#### 5.4 How do sales compare to previous months or weeks

In [ ]:
monthly_revenue_growth = monthly_revenue.pct_change().replace([np.inf, -np.inf], np.nan)
monthly_revenue_growth

In [ ]:
ax = (monthly_revenue_growth * 100).plot.bar(figsize=(10, 6), title="Month-over-Month Revenue Growth")
ax.set(xlabel="Month", ylabel="Growth (%)")
ax.axhline(0, linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

Interpret month-over-month growth only after verifying all twelve months have been loaded successfully. Percentage growth may be undefined when the previous month has zero revenue.

#### 5.5 Which cities are our products delivered to most?


- We will analyse the dataset's delivery addresses and count their frequency in each city.

- This can be accomplished by extracting the city names from the 'Purchase_Address' column and counting the number of times each city appears.

In [ ]:
city_delivery_counts = sales_2019["City"].value_counts()
top_delivery_cities = city_delivery_counts.head(5)
top_delivery_cities

In [ ]:
ax = city_delivery_counts.plot.bar(figsize=(10, 6), title="Sales Line Items by Destination City")
ax.set(xlabel="City", ylabel="Line items")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

This chart counts **sales line items** by destination city, not unique orders or physical deliveries. Interpret the city ranking after running the notebook with the complete data.

#### 5.6 How do product categories compare in revenue generated and quantities ordered?

In [ ]:
def classify_sales_product(product_name: str) -> str:
    """Assign a broad category using the product labels in this dataset."""
    electronics_keywords = ("iphone", "charging cable", "headphones", "monitor", "batteries", "phone")
    return "Electronics" if any(keyword in product_name.lower() for keyword in electronics_keywords) else "Other"

sales_2019["Product_Category"] = sales_2019["Product"].map(classify_sales_product)
category_performance = sales_2019.groupby("Product_Category").agg(revenue=("Sales", "sum"), units_sold=("Quantity_Ordered", "sum"))
category_revenue = category_performance["revenue"]
category_units_sold = category_performance["units_sold"]
category_performance

In [ ]:
for sales_metric, axis_label in (("revenue", "Revenue ($)"), ("units_sold", "Units sold")):
    ax = category_performance[sales_metric].plot.bar(figsize=(9, 5), title=f"Product Category — {axis_label}")
    ax.set(xlabel="Category", ylabel=axis_label)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

Product categories here use a simple keyword-based classification. Compare the resulting revenue and units after running the notebook; verify the categories before drawing business conclusions.

#### 5.7 You are required to show additional details from your findings in your data.

1. Revenue breakdown by category.

- The bar plot of total income by product category shows that the "Electronics" category earns much more money than the "Other" category. This indicates that electronics products are the key revenue drivers for the company.

2. Quantity-Ordered Comparison

- The bar plot depicting total quantity ordered by product category shows a similar tendency, with the "Electronics" category having a significantly higher quantity ordered than the "Other". This means that, while the "Other" category includes a wide range of products, electronics account for the majority of sales volume.

3. Diversification considerations

- While electronics items now account for the majority of income and sales volume, expanding product offerings in the "Other" category may reduce the risks associated with over-reliance on a single product category. Exploring new product lines or expanding existing ones may broaden the client base and improve overall market resilience.

### 5.8 Weekday hypothesis test
- **H0:** Quantity ordered follows the same distribution across weekdays.
- **H1:** At least one weekday differs.

A Kruskal–Wallis test treats weekday as a categorical grouping variable; it does not assume weekday names have a numeric rank.

In [ ]:
# Weekday is categorical, so compare distributions across all seven days.
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_samples = [sales_2019.loc[sales_2019["Day_of_Week"].eq(day), "Quantity_Ordered"].to_numpy() for day in weekday_order]
weekday_samples = [sample for sample in weekday_samples if len(sample)]
weekday_statistic, weekday_p_value = kruskal(*weekday_samples)
significance_level = 0.05
print(f"Kruskal–Wallis H={weekday_statistic:.4f}; p={weekday_p_value:.6g}")
print("Reject H0: at least one weekday distribution differs." if weekday_p_value < significance_level else "Fail to reject H0: insufficient evidence of weekday differences.")

### Interpreting the hypothesis test
A p-value below 0.05 indicates evidence of differences among weekday quantity distributions. Otherwise, the test does not establish that weekday distributions are identical. Consider effect size and practical significance before making operational decisions.

## 6. Deployment

In [ ]:
# Export the clean, enriched dataset for dashboards or downstream analysis.
clean_sales_export_path = OUTPUT_DIR / "sales_2019_cleaned.csv"
sales_2019.to_csv(clean_sales_export_path, index=False)
print(f"Saved {len(sales_2019):,} rows to {clean_sales_export_path.resolve()}")